## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-21-16-53-24 +0000,nypost,OnlyFans model plans to buy mom dream retireme...,https://nypost.com/2026/11/21/us-news/onlyfans...
1,2026-11-21-16-47-23 +0000,startribune,Live: Springfield takes big lead into fourth q...,https://www.startribune.com/live-scores-update...
2,2026-11-21-16-45-00 +0000,nypost,Staggering snow storm leaves California’s skie...,https://nypost.com/2026/11/21/us-news/mammoth-...
3,2026-11-21-16-35-15 +0000,bbc,FBI reportedly investigating apparent hack of ...,https://www.bbc.co.uk/news/articles/cvze7de86p...
4,2026-11-21-16-33-50 +0000,nypost,Wife gushed she was ‘so proud’ of husband in t...,https://nypost.com/2026/11/21/us-news/wife-gus...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2705/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
51,trump,43
47,he,16
59,new,15
693,anthropic,14
514,tech,13


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
307,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...,125
233,2026-11-20-21-35-47 +0000,cbc,Trump threatens to penalize U.S. tech companie...,https://www.cbc.ca/news/world/ai-data-centres-...,104
222,2026-11-20-21-58-36 +0000,latimes,Trump says he'll penalize U.S. companies that ...,https://www.latimes.com/politics/story/2026-11...,97
13,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...,94
149,2026-11-21-01-03-00 +0000,wsj,During President Trump’s White House briefing ...,https://www.wsj.com/politics/elections/how-a-w...,93


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
307,125,2026-11-20-18-06-47 +0000,chitri,Donald Trump says he plans to penalize US tech...,https://www.chicagotribune.com/2026/11/20/trum...
28,70,2026-11-21-15-10-50 +0000,chitri,Mexico detains top cartel boss ‘El Chapo Isidr...,https://www.chicagotribune.com/2026/11/21/el-c...
283,48,2026-11-20-19-39-00 +0000,wsj,"Sen. Tom Cotton (R., Ark.), a China hawk, tell...",https://www.wsj.com/politics/national-security...
277,48,2026-11-20-19-48-40 +0000,nypost,Anthropic’s new IPO millionaires awkward about...,https://nypost.com/2026/11/20/business/anthrop...
236,45,2026-11-20-21-21-40 +0000,cbc,Former White House press secretary Katie Zacha...,https://www.cbc.ca/news/world/katie-zacharia-c...
13,43,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...
4,40,2026-11-21-16-33-50 +0000,nypost,Wife gushed she was ‘so proud’ of husband in t...,https://nypost.com/2026/11/21/us-news/wife-gus...
261,38,2026-11-20-20-25-49 +0000,nypost,"‘GTA 6’ maker Take-Two rakes in nearly $4B, sh...",https://nypost.com/2026/11/20/business/take-tw...
165,35,2026-11-21-00-25-49 +0000,cbc,U.S. senator says Canada should make interim t...,https://www.cbc.ca/news/politics/us-canada-int...
297,34,2026-11-20-18-48-55 +0000,nypost,Senate approves bipartisan bill to force data ...,https://nypost.com/2026/11/20/us-news/senate-a...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
